# Figure 2 — Failure to Learn the Optimal Velocity Field

**Goal:** Replicate Figure 2 on toy datasets.

1. **Left panel:** Average error $\mathbb{E}\|u_\theta(x_t, t) - \hat{u}^\star(x_t, t)\|^2$ vs $t$, for varying dataset sizes.
2. **Right panel:** Nearest-neighbor distance between generated and training samples.
3. **Visualization:** Scatter plots of generated vs training data.

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/YOUR_REPO/ClosedFlowMatching.git 2>/dev/null || true
    os.chdir('ClosedFlowMatching')
    !pip install -q -r requirements.txt
    sys.path.insert(0, '.')
else:
    sys.path.insert(0, '..')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

from src.data.toy import ToyDataset
from src.models.mlp import VelocityMLP
from src.flow_matching.cfm import CFMTrainer
from src.flow_matching.sampler import ode_sample
from src.metrics.evaluation import velocity_approximation_error, nearest_neighbor_distance

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False
})
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 2.1 Train models for different dataset sizes

We fix the network capacity and vary the number of training samples.
As the dataset grows, $\hat{u}^\star$ becomes more complex and harder to approximate.

In [ ]:
n_samples_list = [10, 50, 200, 1000, 5000]
n_steps = 15000
batch_size = 256

models = {}
datasets = {}

for n_samp in n_samples_list:
    print(f'\n--- Training with n = {n_samp} ---')
    ds = ToyDataset('gaussian_mixture', n_samples=n_samp, seed=42)
    datasets[n_samp] = ds

    model = VelocityMLP(data_dim=2, hidden_dim=256, n_layers=4)
    trainer = CFMTrainer(model, lr=1e-3, device=device)
    trainer.train(ds, n_steps=n_steps, batch_size=min(batch_size, n_samp), log_every=5000)
    models[n_samp] = model

print('\nAll models trained.')

## 2.2 Left panel: velocity approximation error vs time

In [ ]:
t_grid = torch.linspace(0.01, 0.99, 50)
colors = cm.viridis(np.linspace(0.15, 0.95, len(n_samples_list)))

fig, ax = plt.subplots(figsize=(8, 5))

for i, n_samp in enumerate(n_samples_list):
    data_flat = datasets[n_samp].data
    errs = velocity_approximation_error(
        models[n_samp], data_flat, t_grid, n_eval=256, device=device
    )
    ts = sorted(errs.keys())
    vals = [errs[t] for t in ts]
    ax.plot(ts, vals, color=colors[i], linewidth=2, label=f'n = {n_samp}')

ax.set_xlabel('t')
ax.set_ylabel(r'$\mathbb{E}\|u_\theta - \hat{u}^\star\|^2$')
ax.set_title('Velocity approximation error')
ax.legend()
plt.tight_layout()
plt.savefig('fig2_left_velocity_error.pdf', bbox_inches='tight')
plt.show()

## 2.3 Right panel: nearest-neighbor distance (memorization metric)

In [ ]:
n_gen = 500

fig, ax = plt.subplots(figsize=(7, 5))
nn_means = []

for n_samp in n_samples_list:
    gen = ode_sample(models[n_samp], n_gen, (2,), n_steps=100, device=device).cpu()
    train_data = datasets[n_samp].data
    dists = nearest_neighbor_distance(gen, train_data)
    nn_means.append(dists.mean().item())

ax.bar(range(len(n_samples_list)), nn_means, color='steelblue', edgecolor='white')
ax.set_xticks(range(len(n_samples_list)))
ax.set_xticklabels([str(n) for n in n_samples_list])
ax.set_xlabel('# training samples')
ax.set_ylabel('Mean NN distance')
ax.set_title('Nearest-neighbor distance (memorization)')
plt.tight_layout()
plt.savefig('fig2_right_nn_distance.pdf', bbox_inches='tight')
plt.show()

## 2.4 Visualize generated vs training data

In [ ]:
fig, axes = plt.subplots(1, len(n_samples_list), figsize=(4 * len(n_samples_list), 4))

for i, n_samp in enumerate(n_samples_list):
    ax = axes[i]
    train = datasets[n_samp].data.numpy()
    gen = ode_sample(models[n_samp], 500, (2,), n_steps=100, device=device).cpu().numpy()

    ax.scatter(train[:, 0], train[:, 1], s=3, alpha=0.3, c='steelblue', label='Train')
    ax.scatter(gen[:, 0], gen[:, 1], s=3, alpha=0.5, c='tomato', label='Generated')
    ax.set_title(f'n = {n_samp}')
    ax.set_aspect('equal')
    ax.legend(markerscale=3, fontsize=8)

plt.tight_layout()
plt.savefig('fig2_scatter_comparison.pdf', bbox_inches='tight')
plt.show()